In [2]:
import logging
import os
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Silence Git warning logs from MLflow
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

import mlflow
import mlflow.sklearn

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------------
# STEP 1: Data Preparation
# -------------------------------------------------------------------------
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
df["date_time"] = pd.to_datetime(df["date_time"])

df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

weather_dummies = pd.get_dummies(
    df["weather_main"], prefix="weather", drop_first=True
)
df = pd.concat([df, weather_dummies], axis=1)

feature_cols = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
] + list(weather_dummies.columns)

X = df[feature_cols]
y_reg = df["traffic_volume"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------------
# STEP 2: MLflow Experiment Tracking Execution
# -------------------------------------------------------------------------
mlflow.set_experiment("Traffic_Volume_Regression_Suite")
logger.info("MLflow experiment initialized: Traffic_Volume_Regression_Suite")

# Run 1: Ridge Baseline
with mlflow.start_run(run_name="Ridge_Baseline"):
    alpha = 1.0
    reg_ridge = Ridge(alpha=alpha)
    reg_ridge.fit(X_train_scaled, y_train)
    preds = reg_ridge.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", alpha)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Use cloudpickle serialization format to avoid skops untrusted type checks
    mlflow.sklearn.log_model(
        sk_model=reg_ridge,
        name="ridge_model",
        serialization_format="cloudpickle",
    )
    logger.info(f"Logged Ridge Run -> MAE: {mae:.2f}, R2: {r2:.4f}")

# Run 2: Random Forest Ensemble (Fixed logging call)
with mlflow.start_run(run_name="RandomForest_Ensemble"):
    n_estimators = 100
    max_depth = 12
    reg_rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1,
    )
    reg_rf.fit(X_train, y_train)
    preds_rf = reg_rf.predict(X_test)

    mae_rf = mean_absolute_error(y_test, preds_rf)
    r2_rf = r2_score(y_test, preds_rf)

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("mae", mae_rf)
    mlflow.log_metric("r2", r2_rf)

    # Explicitly trust sklearn tree types or specify cloudpickle
    mlflow.sklearn.log_model(
        sk_model=reg_rf,
        name="rf_model",
        serialization_format="cloudpickle",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )
    logger.info(f"Logged Random Forest Run -> MAE: {mae_rf:.2f}, R2: {r2_rf:.4f}")

print("\n" + "=" * 65)
print("TASK 4: MLFLOW EXPERIMENT TRACKING SUMMARY")
print("=" * 65)
summary_table = pd.DataFrame(
    {
        "Run Name": ["Ridge_Baseline", "RandomForest_Ensemble"],
        "Parameters Logged": ["alpha=1.0", "n_estimators=100, max_depth=12"],
        "MAE (vehicles/hr)": [round(mae, 2), round(mae_rf, 2)],
        "R-Squared (R²)": [round(r2, 4), round(r2_rf, 4)],
        "Artifact Saved": ["ridge_model", "rf_model"],
    }
)
print(summary_table.to_string(index=False))

2026-10-05 17:52:27,545 - INFO - MLflow experiment initialized: Traffic_Volume_Regression_Suite


2026/10/05 17:52:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-10-05 17:52:30,678 - INFO - Logged Ridge Run -> MAE: 823.00, R2: 0.7199


2026/10/05 17:52:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-10-05 17:52:38,109 - INFO - Logged Random Forest Run -> MAE: 259.68, R2: 0.9461

TASK 4: MLFLOW EXPERIMENT TRACKING SUMMARY
             Run Name              Parameters Logged  MAE (vehicles/hr)  R-Squared (R²) Artifact Saved
       Ridge_Baseline                      alpha=1.0             823.00          0.7199    ridge_model
RandomForest_Ensemble n_estimators=100, max_depth=12             259.68          0.9461       rf_model
